# Local FM Arrow to SDM

Provide a validated bundle, workload, compatible DiskGraph index path, and pinned SDM environment described in this folder's README. This notebook samples once and reuses the same Arrow batch for local inference and both HTTP bodies.


In [1]:
import os
from pathlib import Path
import torch
from kumo import fm
from adapter import arrow_to_sdm
from sdm.models import KumoRelational

fixture = Path(os.environ["FM_SDM_FIXTURE"])
torch.set_num_threads(2)
bundle_path = Path(os.environ.get("FM_SDM_BUNDLE", str(fixture / "bundle")))
with fm.prepare(bundle_path, fixture / "workload.json",
                index=fixture / "fm-index", threads=2) as loader:
    with loader.materialize() as request:
        batch = request.load()
print(batch.tables("context")["instance_table"].schema)
print(batch.logical_arrow_bytes)


instance_id: int64 not null
__fm_entity_key: int64 not null
__fm_anchor: timestamp[us, tz=UTC] not null
__fm_target: string not null
584


In [2]:
inputs = arrow_to_sdm(batch)
arrays_body = batch.encode("arrays")
ipc_body = batch.encode("arrow_ipc")
print({"arrays_bytes": len(arrays_body), "ipc_bytes": len(ipc_body)})


{'arrays_bytes': 3374, 'ipc_bytes': 8859}


In [3]:
model = KumoRelational(task="classification", device="cpu")
with torch.inference_mode():
    one_shot = model(**inputs, num_estimators=1,
                     generator=torch.Generator().manual_seed(42), num_hops=2)
    model.fit(inputs["x_context"], inputs["y_context"],
              inputs["related_context_tables"], num_estimators=1,
              generator=torch.Generator().manual_seed(42), num_hops=2)
    reused_context = model.predict(inputs["x_query"], inputs["related_query_tables"])
torch.testing.assert_close(one_shot.numerical, reused_context.numerical,
                           rtol=1e-5, atol=1e-5)
print(one_shot.to_arrow().schema)
print("one-shot/fit-predict max absolute difference:",
      float((one_shot.numerical - reused_context.numerical).abs().max()))


yes: float
one-shot/fit-predict max absolute difference: 0.0
